Load training data

In [1]:
import idx2numpy
import numpy as np

image_path = 'MNIST_ORG/train-images.idx3-ubyte'
label_path = 'MNIST_ORG/train-labels.idx1-ubyte'
train_images = idx2numpy.convert_from_file(image_path)
train_labels = idx2numpy.convert_from_file(label_path)

train_images = train_images.reshape(train_images.shape[0], -1) / 255.0
train_labels = np.eye(10)[train_labels]



# 'train_images' is now a properly shaped NumPy array

In [2]:
train_labels = train_labels.astype(np.int32) # Convert labels to one-hot encoding
print(train_labels[:5])

[[0 0 0 0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1]]


Training process

In [3]:
from functions import *


In [4]:
input_dim = 784
hidden_dim = [64, 64]
output_dim = 10
lr = 0.01
batch_size = 64
W, b = initwb(input_dim, hidden_dim, output_dim)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

def train(epochs, W, b, X_train, y_train, lr, batch_size):
    losses = []  # To store loss per epoch
    
    for epoch in range(epochs):
        # Shuffle data (optional but recommended)
        indices = np.random.permutation(X_train.shape[0])
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]
        
        epoch_loss = 0
        num_batches = X_train.shape[0] // batch_size
        
        for i in range(num_batches):
            start = i * batch_size
            end = start + batch_size
            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]
            
            # Forward pass
            activations = forward_pass(X_batch, W, b)
            
            # Compute loss
            loss = multiple_cross_entropy_loss(y_batch, activations[-1])
            epoch_loss += loss
            
            # Backward pass
            dW, db = backward_propagation(y_batch, W, b, activations)
            # Update parameters
            update_parameters(W, b, dW, db, lr)
        
        # Average loss for the epoch
        avg_loss = epoch_loss / num_batches
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

# Run training and collect losses
epochs = 50 
losses = train(epochs, W, b, train_images, train_labels, lr, batch_size)

# Now animate the losses
fig, ax = plt.subplots()
ax.set_xlim(0, epochs)
ax.set_ylim(0, max(losses) * 1.1)  # Adjust y-limit based on max loss
line, = ax.plot([], [], 'r-')

def animate(frame):
    line.set_data(range(frame+1), losses[:frame+1])
    return line,

ani = FuncAnimation(fig, animate, frames=epochs, interval=200, blit=True)
"""
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.show()
"""
from IPython.display import HTML
HTML(ani.to_jshtml())
# The animation doesn't work actually, but it does give a graph of the losses

AttributeError: 'list' object has no attribute 'shape'

Testing stage


In [ ]:
test_image_path = 'MNIST_ORG/t10k-images.idx3-ubyte'
test_label_path = 'MNIST_ORG/t10k-labels.idx1-ubyte'
test_images = idx2numpy.convert_from_file(test_image_path)
test_labels = idx2numpy.convert_from_file(test_label_path)

test_images = test_images.reshape(test_images.shape[0], -1) / 255.0
test_labels = np.eye(10)[test_labels]


In [ ]:
y_test_pred = forward_pass(test_images, W, b)[-1]
test_accuracy = calculate_accuracy(test_labels, y_test_pred)
print(f"Test accuracy: {test_accuracy:.2f}%")

Test accuracy: 97.27%


Save model

In [ ]:
from functions import save_model
save_model(W, b, "mnist_model.pkl")